# Deformable stage on one validation slab

The final model predicts the shift along the phase-encoding (PE) axis for one real-distortion validation slab; the
T1 is resampled through the exact rigid transform. Printed: the slab's TRE without and with the predicted shift.
Shown: T1 edges over the slab, without and with the predicted shift.

Needs the data (`dlir_data/` in the repository root or `DLIR_DATA`) and the weights: `runs/deformable/00_final`
from `run_all.sh`, otherwise they are downloaded from W&B.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))                   # scripts/: slabsim and deformable
import matplotlib.pyplot as plt
import torch

from deformable.data import REPO, ValSlabs
from deformable.evaluate import load_run
from deformable.losses import voxel_errors
from deformable.model import predict, warp

RUN = REPO / 'runs' / 'deformable' / '00_final'
if not (RUN / 'best.pt').exists():
    import wandb
    wandb.Api().artifact('WANDB_ENTITY/dlir-deformable/00_final-model:latest').download(str(RUN))
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net, cfg = load_run(RUN, dev)

In [ ]:
SLAB = '195950_rfMRI_REST1_LR/t32_p30_small'                    # inferior axial slab, 32 mm
ds = ValSlabs()
b = ds[[f'{d.parent.name}/{d.name}' for d in ds.dirs].index(SLAB)]
with torch.no_grad():
    u, _ = predict(net, cfg, b['x'], b['pe_axis'], b['pe_sign'])
    t1_moved = warp(b['x'][None, 1:].to(dev), u)[0, 0].cpu()      # T1 at the predicted positions
print(f"TRE without deformation {voxel_errors(torch.zeros_like(u), b, b['M']).mean():.2f} mm, "
      f"with the predicted shift {voxel_errors(u, b, b['M']).mean():.2f} mm")

In [ ]:
k = b['x'].shape[3] // 2                                       # middle slice of the slab
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, t1, title in zip(axes, [b['x'][1], t1_moved], ['without deformation', 'with the predicted shift']):
    ax.imshow(b['x'][0, :, :, k].T, cmap='gray', origin='lower', vmin=0, vmax=1)
    ax.contour(t1[:, :, k].T, levels=[0.7], colors='yellow', linewidths=0.8)   # grey / white matter boundary
    ax.set_title(f'{SLAB}: {title}', fontsize=9)
    ax.axis('off')
plt.show()